# Assignment 3: Supervisor Pattern - Travel Planning System

**Pattern:** A **Supervisor Agent** coordinates 3 specialized sub-agents:
1. **Flight Agent** - searches and books flights
2. **Hotel Agent** - searches and books hotels  
3. **Activity Agent** - finds and books local tours

**Why one agent isn't enough:**
- Each domain (airlines, hotels, activities) uses different APIs and data formats
- One agent with all tools would have a huge prompt and be hard to maintain
- The supervisor splits work, keeps each agent focused, and combines results

## 1. Setup & Imports

In [37]:
import os
from datetime import datetime
from dotenv import load_dotenv
from langchain_google_genai import ChatGoogleGenerativeAI
from langchain_core.messages import HumanMessage
from langchain_core.tools import tool
from langgraph.prebuilt import create_react_agent

load_dotenv(os.path.join(os.getcwd(), '..', '.env'))

model = ChatGoogleGenerativeAI(
    model="gemini-2.5-flash",
    google_api_key=os.getenv("GOOGLE_API_KEY"),
)

print("Model initialized successfully")

Model initialized successfully


## 2. Define API Tools (Stubs)

These simulate real booking APIs with hardcoded responses.

In [38]:
@tool
def search_flights(origin: str, destination: str, date: str) -> str:
    """Search available flights between two airports on a given date (ISO format)."""
    return (
        f"Found 3 flights from {origin} to {destination} on {date}:\n"
        f"  1. SV-201  06:00-08:30  SAR 850  (non-stop)\n"
        f"  2. SV-305  12:15-14:45  SAR 720  (non-stop)\n"
        f"  3. EK-412  09:00-15:20  SAR 650  (1 stop DXB)"
    )

@tool
def book_flight(flight_number: str, passenger_name: str) -> str:
    """Book a specific flight for a passenger."""
    return f"Flight {flight_number} booked for {passenger_name}. Confirmation: TRV-{hash(flight_number) % 10000:04d}"

@tool
def search_hotels(city: str, checkin: str, checkout: str) -> str:
    """Search available hotels in a city for the given check-in/check-out dates."""
    return (
        f"Found 3 hotels in {city} ({checkin} to {checkout}):\n"
        f"  1. Hilton Garden Inn      - SAR 450/night ****\n"
        f"  2. Marriott City Center   - SAR 620/night *****\n"
        f"  3. Holiday Inn Express    - SAR 310/night ***"
    )

@tool
def book_hotel(hotel_name: str, guest_name: str, checkin: str, checkout: str) -> str:
    """Book a hotel room for a guest."""
    return f"{hotel_name} booked for {guest_name} ({checkin} to {checkout}). Confirmation: HTL-{hash(hotel_name) % 10000:04d}"

@tool
def search_activities(city: str, date: str) -> str:
    """Search tours and activities available in a city on a given date."""
    return (
        f"Found 3 activities in {city} on {date}:\n"
        f"  1. Old Town Walking Tour      - SAR 120  (3 hrs, 09:00)\n"
        f"  2. Desert Safari & BBQ Dinner  - SAR 280  (6 hrs, 15:00)\n"
        f"  3. Local Food Tasting Tour     - SAR 95   (2 hrs, 12:00)"
    )

@tool
def book_activity(activity_name: str, participant_name: str, date: str) -> str:
    """Book an activity for a participant."""
    return f"{activity_name} booked for {participant_name} on {date}. Confirmation: ACT-{hash(activity_name) % 10000:04d}"

print("6 booking tools defined")

6 booking tools defined


## 3. Create Sub-Agents

Each sub-agent is a **ReAct agent** specialized in one domain.

In [39]:
today = datetime.now().strftime('%Y-%m-%d')

flight_agent = create_react_agent(
    model=model,
    tools=[search_flights, book_flight],
    prompt=(
        "You are a flight booking specialist. "
        "Parse natural language travel requests into structured flight searches. "
        "Convert relative dates (e.g. 'next Friday') into ISO format based on "
        f"today's date: {today}. "
        "Always search for available flights first, then book the best option "
        "unless the user specifies a preference."
    ),
)

hotel_agent = create_react_agent(
    model=model,
    tools=[search_hotels, book_hotel],
    prompt=(
        "You are a hotel booking specialist. "
        "Parse natural language accommodation requests into structured searches. "
        "Convert relative dates into ISO format based on "
        f"today's date: {today}. "
        "Always search for available hotels first, then book the best value "
        "option unless the user specifies a preference."
    ),
)

activity_agent = create_react_agent(
    model=model,
    tools=[search_activities, book_activity],
    prompt=(
        "You are a local activities and tours specialist. "
        "Help users find and book tours, excursions, and activities at their "
        "destination. Convert relative dates into ISO format based on "
        f"today's date: {today}. "
        "Search for activities first, then recommend the best option."
    ),
)

print("3 sub-agents created: flight_agent, hotel_agent, activity_agent")

3 sub-agents created: flight_agent, hotel_agent, activity_agent


C:\Users\Yaz00\AppData\Local\Temp\ipykernel_27188\3518081720.py:3: LangGraphDeprecatedSinceV10: create_react_agent has been moved to `langchain.agents`. Please update your import to `from langchain.agents import create_agent`. Deprecated in LangGraph V1.0 to be removed in V2.0.
  flight_agent = create_react_agent(
C:\Users\Yaz00\AppData\Local\Temp\ipykernel_27188\3518081720.py:16: LangGraphDeprecatedSinceV10: create_react_agent has been moved to `langchain.agents`. Please update your import to `from langchain.agents import create_agent`. Deprecated in LangGraph V1.0 to be removed in V2.0.
  hotel_agent = create_react_agent(
C:\Users\Yaz00\AppData\Local\Temp\ipykernel_27188\3518081720.py:29: LangGraphDeprecatedSinceV10: create_react_agent has been moved to `langchain.agents`. Please update your import to `from langchain.agents import create_agent`. Deprecated in LangGraph V1.0 to be removed in V2.0.
  activity_agent = create_react_agent(


## 4. Wrap Sub-Agents as Tools + Create Supervisor

The sub-agents are wrapped as tools so the **Supervisor** can delegate tasks to them.

In [40]:
@tool
def handle_flights(request: str) -> str:
    """Search and book flights using natural language."""
    result = flight_agent.invoke({"messages": [HumanMessage(content=request)]})
    return result["messages"][-1].content

@tool
def handle_hotels(request: str) -> str:
    """Search and book hotel accommodations using natural language."""
    result = hotel_agent.invoke({"messages": [HumanMessage(content=request)]})
    return result["messages"][-1].content

@tool
def handle_activities(request: str) -> str:
    """Search and book local tours and activities using natural language."""
    result = activity_agent.invoke({"messages": [HumanMessage(content=request)]})
    return result["messages"][-1].content

supervisor = create_react_agent(
    model=model,
    tools=[handle_flights, handle_hotels, handle_activities],
    prompt=(
        "You are a personal travel planning assistant. "
        "You can search and book flights, hotels, and local activities. "
        "Break down user requests into appropriate tool calls and coordinate "
        "the results into a unified travel itinerary. "
        "When a request involves multiple actions, use multiple tools in sequence "
        "and present a clear summary at the end."
    ),
)

print("Supervisor agent created with 3 sub-agent tools")

Supervisor agent created with 3 sub-agent tools


C:\Users\Yaz00\AppData\Local\Temp\ipykernel_27188\3250181762.py:19: LangGraphDeprecatedSinceV10: create_react_agent has been moved to `langchain.agents`. Please update your import to `from langchain.agents import create_agent`. Deprecated in LangGraph V1.0 to be removed in V2.0.
  supervisor = create_react_agent(


## 5. Demo: Simple Request (Flight Only)

In [41]:
print("=" * 70)
print("[1] Simple request: flight only")
print("=" * 70)

response = supervisor.invoke({
    "messages": [HumanMessage(
        content="Find me a flight from Riyadh (RUH) to Jeddah (JED) next Thursday."
    )]
})
print(response["messages"][-1].content)

[1] Simple request: flight only
I found 3 flights from Riyadh to Jeddah on March 19, 2026:
1. SV-201 at 06:00, arriving 08:30, for SAR 850 (non-stop)
2. SV-305 at 12:15, arriving 14:45, for SAR 720 (non-stop)
3. EK-412 at 09:00, arriving 15:20, for SAR 650 (1 stop in DXB)

Which flight would you like to book?


## 6. Demo: Complex Request (Full Trip Planning)

In [42]:
import time
time.sleep(30)  # wait to avoid rate limit on free API

print("=" * 70)
print("[2] Complex request: full trip planning")
print("=" * 70)

response = supervisor.invoke({
    "messages": [HumanMessage(
        content=(
            "Plan a weekend trip for Yazeed: "
            "fly from Riyadh to Jeddah next Thursday, "
            "book a hotel for 2 nights, "
            "and find a fun activity for Friday."
        )
    )]
})
print(response["messages"][-1].content)

[2] Complex request: full trip planning
[{'type': 'text', 'text': "Your trip for Yazeed has been planned!\n\nHere's a summary:\n\n**Flights:**\n*   **From:** Riyadh\n*   **To:** Jeddah\n*   **Date:** Next Thursday, March 20, 2026\n*   **Options:**\n    *   SV-201: 06:00-08:30, SAR 850 (non-stop)\n    *   SV-305: 12:15-14:45, SAR 720 (non-stop)\n    *   EK-412: 09:00-15:20, SAR 650 (1 stop DXB)\n\n**Hotel:**\n*   **Location:** Jeddah\n*   **Hotel:** Holiday Inn Express (3 stars)\n*   **Check-in:** Next Thursday, March 20, 2026\n*   **Nights:** 2\n*   **Price:** SAR 310/night\n\n**Activity:**\n*   **What:** Desert Safari & BBQ Dinner\n*   **Where:** Jeddah\n*   **When:** Friday, March 21, 2026, at 15:00 (6 hours)\n*   **Price:** SAR 280\n\nWould you like me to proceed with booking these options, or would you like to explore other choices?", 'extras': {'signature': 'CvIFAb4+9vs6Qb/QFsXHrP4GXvYdBHUdu7hFlctM9aqsrYxaPM1BxwIxnd6a0i86uZ7tQcKFTJUTNXgbcgRaHiqg5hfJGtIR8wGhH6zFIBMWuLMFe3f4uN8fIl1v